# Deterministic rolling-horizon EMS for REINFORCE Lab 2

## What this code does
```
Decides, for every 15-min period, how much energy Lab 2 should buy from and
sell to the grid, charge into and discharge from a shared battery, and which
fraction of the PV panels to keep in operation, so as to maximise
real profit - battery degradation cost.

1. Reads PV, demand and spot price from Data/lab2_2025.xlsx and builds
   PB = spot + c and PS = spot.
2. For each roll r = 1, ..., N_ROLLS:
     a. takes the 7-day window starting at START_DATE + (r - 1),
     b. builds the optimisation model (objective + constraints (1)-(9)),
     c. solves it with SCIP,
     d. keeps only the first day of the solution as the implemented decision,
     e. passes the SOC at the end of that day to the next window.
3. Writes the implemented decisions and a daily summary to Results/.
```

## Assumptions
```
1) The buildings of Lab 2 are connected: they form one node with total PV and
   total demand, one shared battery and one grid connection, and face the same
   purchase and selling prices. Energy is shared between buildings without
   losses or fees.
2) Deterministic model with perfect foresight: PV, demand and prices are known
   for the whole window.
3) PS_t = spot_t (AT day-ahead) and PB_t = spot_t + c, with c a constant.
4) Arbitrage is allowed: the battery may charge from the grid and sell later.
   PV energy can be used on site, stored or sold.
5) PV panels can be partially switched off (F_t). In extreme cases (battery
   full, selling not possible or not profitable, very high PV) part of the PV
   production is "thrown away".
6) Demand must always be met.
7) Using the battery has a cost: every cycle consumes battery life, and fast
   charging/discharging or frequently exceeding the preservation limits
   (MinP, MaxP) make it age faster. Cost = DEG * PM / MaxC.
8) Terminal condition: SOC at the end of every window >= SOC_INIT
   (the initial SOC of the first window).
9) Time: 15-min periods, all energies in kWh per period, CET (no DST).
10) No electrolyzer.
```

## 1. Settings and parameters
Change the dates and the number of days here. All parameters are assumptions and can be adjusted.

In [ ]:
from pathlib import Path
import time
import pandas as pd
import pyomo.environ as pyo

START_DATE = "2025-10-01"    # first implemented day (YYYY-MM-DD, CET)
N_ROLLS = 7                  # number of rolls = number of implemented days
WINDOW_DAYS = 7              # length of each optimisation window [days]

In [2]:
# Prices
C_MARKUP = 0.12      # c: grid fees + taxes added to spot for buying   [EUR/kWh]
#                      PB_t = spot_t + c ,  PS_t = spot_t

# Battery size and limits
MAXB = 60.0          # MaxB: battery capacity                          [kWh]
SOC_INIT = 0.5       # SOC_init: initial SOC of the first window       [0,1]
SOC_MIN = 0.05       # physical minimum SOC                            [0,1]
SOC_MAX = 0.95       # physical maximum SOC                            [0,1]
MINP = 0.15          # MinP: minimum SOC to preserve the battery       [0,1]
MAXP = 0.85          # MaxP: maximum SOC to preserve the battery       [0,1]
CD = 7.5             # CD: max charge/discharge per period             [kWh/period]

# Efficiency and losses
ALPHA = 0.95         # alpha: charge efficiency
BETA = 0.95          # beta: discharge efficiency
ETA = 1e-5           # eta: natural loss per period (~3 %/month)

# Degradation
PC = 2 * MAXB        # PC: throughput (in + out) for 1 equivalent cycle [kWh/cycle]
MAXC = 6000          # MaxC: theoretical total life                    [cycles]
PM = 400 * MAXB      # PM: price of the battery                        [EUR]
FC = 0.2             # FC: fast-charging penalty coefficient
CP1 = 0.5            # CP_1: linear cost of exceeding the preservation range
CP2 = 5.0            # CP_2: quadratic cost of exceeding the preservation range
# BH_INIT = 5000     # optional: remaining battery life [cycles], see constraint (4b)

# Grid
G = 25.0             # G: max purchase + sale per period               [kWh/period]

# Solver
SOLVER_OPTIONS = {"limits/gap": 1e-4,         # relative optimality gap
                  "limits/time": 600,         # time limit per window [s]
                  "numerics/feastol": 1e-7}   # tighter feasibility tolerance (quadratic terms)

PPD = 96             # periods per day (15 min)

## 2. Data
Reads `Data/lab2_2025.xlsx` (PV, demand and spot price per 15 min) and builds $PB_t = spot_t + c$ and $PS_t = spot_t$.

In [3]:
HERE = Path.cwd()          # folder of this notebook
DATA_FILE = HERE / "Data" / "lab2_2025.xlsx"
RESULTS_DIR = HERE / "Results"

data = pd.read_excel(DATA_FILE, sheet_name="data")
data["PB"] = data["spot"] + C_MARKUP
data["PS"] = data["spot"]

start_idx = data.index[data["datetime"] == pd.Timestamp(START_DATE)]
if len(start_idx) == 0:
    raise ValueError(f"START_DATE {START_DATE} not found in the data")
start_idx = int(start_idx[0])
last_needed = start_idx + (N_ROLLS - 1 + WINDOW_DAYS) * PPD
if last_needed > len(data):
    raise ValueError("Not enough data: reduce N_ROLLS or WINDOW_DAYS, or start earlier")

data.head()

,datetime,day,period,PV,D,spot,PB,PS
0,2025-01-01 00:00:00,1,1,0.0,0.518,0.10900,0.22900,0.10900
1,2025-01-01 00:15:00,1,2,0.0,0.256,0.10900,0.22900,0.10900
2,2025-01-01 00:30:00,1,3,0.0,0.159,0.10900,0.22900,0.10900
3,2025-01-01 00:45:00,1,4,0.0,0.209,0.10900,0.22900,0.10900
4,2025-01-01 01:00:00,1,5,0.0,0.406,0.09703,0.21703,0.09703


## 3. Model for one window

For a window with periods $t = 1,\dots,T$ ($T = 96 \cdot$ `WINDOW_DAYS`):

**Objective** (real profit $-$ battery degradation cost)
$$\max \; \sum_{t} \left(PS_t\, S_t - PB_t\, B_t\right) \;-\; \frac{DEG}{MaxC}\, PM$$

**Constraints**

(1) Physical SOC limits: $\; SOC_{min} \le SOC_t \le SOC_{max}$

(2) SOC recursion: $\; SOC_t = (1-\eta)\, SOC_{t-1} + \dfrac{\alpha I_t - O_t/\beta}{MaxB}$, with $SOC_0$ = initial SOC of the window

(3) Energy balance: $\; F_t\, PV_t + O_t + B_t = D_t + I_t + S_t$

(4) Degradation [cycles]:
$$DEG = \sum_t \frac{I_t + O_t}{PC} + \sum_t \frac{FC\,(I_t^2 + O_t^2)}{CD \cdot PC} + \sum_t \left[CP_1 (QU_t + QL_t) + CP_2 (QU_t^2 + QL_t^2)\right] \frac{MaxB}{PC}$$
(optional (4b): $DEG \le BH_{init}$, commented out)

(5) Preservation range: $\; SOC_t \le MaxP + QU_t, \quad SOC_t \ge MinP - QL_t$

(6) Max charge/discharge, no simultaneous charge and discharge: $\; I_t \le CD\, C_t, \quad O_t \le CD\,(1 - C_t)$

(7) Range of $F_t$: $\; 0 \le F_t \le 1$

(8) Max purchase + sale: $\; B_t + S_t \le G$

(9) Terminal condition: $\; SOC_T \ge SOC_{init}$ (initial SOC of the first window)

**Variables**: $B_t, S_t, I_t, O_t, QU_t, QL_t \ge 0$; $F_t \in [0,1]$; $C_t \in \{0,1\}$; $SOC_t$.
The model is a mixed-integer quadratic program (MIQP), solved with SCIP.

In [4]:
def build_model(win, soc0):
    """win: DataFrame with the window's periods; soc0: SOC at the start of the window."""
    m = pyo.ConcreteModel()
    n = len(win)
    m.T = pyo.RangeSet(1, n)

    PV = dict(zip(m.T, win["PV"]))
    D = dict(zip(m.T, win["D"]))
    PB = dict(zip(m.T, win["PB"]))
    PS = dict(zip(m.T, win["PS"]))

    # Decision variables
    m.B = pyo.Var(m.T, within=pyo.NonNegativeReals)     # purchase   [kWh]
    m.S = pyo.Var(m.T, within=pyo.NonNegativeReals)     # sale       [kWh]
    m.I = pyo.Var(m.T, within=pyo.NonNegativeReals)     # charge     [kWh]
    m.O = pyo.Var(m.T, within=pyo.NonNegativeReals)     # discharge  [kWh]
    m.F = pyo.Var(m.T, within=pyo.NonNegativeReals)     # fraction of panels in operation
    m.C = pyo.Var(m.T, within=pyo.Binary)               # 1 = charge, 0 = discharge
    m.SOC = pyo.Var(m.T)                                # SOC at end of period t
    m.QU = pyo.Var(m.T, within=pyo.NonNegativeReals)    # excess above MaxP [SOC fraction]
    m.QL = pyo.Var(m.T, within=pyo.NonNegativeReals)    # excess below MinP [SOC fraction]

    # (4) Degradation DEG: equivalent cycles of battery life consumed [cycles]
    #     = cycling (throughput)  + fast charging (quadratic in I, O)
    #     + exceeding the preservation range (linear and quadratic in QU, QL)
    m.DEG = pyo.Expression(expr=
        sum(m.I[t] + m.O[t] for t in m.T) / PC
        + sum(FC * (m.I[t] ** 2 + m.O[t] ** 2) for t in m.T) / (CD * PC)
        + sum(CP1 * (m.QU[t] + m.QL[t]) + CP2 * (m.QU[t] ** 2 + m.QL[t] ** 2) for t in m.T) * MAXB / PC
    )

    # Objective: real profit - battery degradation cost
    m.obj = pyo.Objective(
        expr=sum(m.S[t] * PS[t] - m.B[t] * PB[t] for t in m.T) - m.DEG * PM / MAXC,
        sense=pyo.maximize)

    # (1) physical SOC limits
    m.c1_max = pyo.Constraint(m.T, rule=lambda m, t: m.SOC[t] <= SOC_MAX)
    m.c1_min = pyo.Constraint(m.T, rule=lambda m, t: m.SOC[t] >= SOC_MIN)

    # (2) SOC recursion
    def soc_rule(m, t):
        prev = soc0 if t == 1 else m.SOC[t - 1]
        return m.SOC[t] == (1 - ETA) * prev + (ALPHA * m.I[t] - m.O[t] / BETA) / MAXB
    m.c2 = pyo.Constraint(m.T, rule=soc_rule)

    # (3) energy balance
    m.c3 = pyo.Constraint(m.T, rule=lambda m, t:
                          m.F[t] * PV[t] + m.O[t] + m.B[t] == D[t] + m.I[t] + m.S[t])

    # (4b) optional: remaining battery life
    # m.c4b = pyo.Constraint(expr=m.DEG <= BH_INIT)

    # (5) preservation range
    m.c5_max = pyo.Constraint(m.T, rule=lambda m, t: m.SOC[t] <= MAXP + m.QU[t])
    m.c5_min = pyo.Constraint(m.T, rule=lambda m, t: m.SOC[t] >= MINP - m.QL[t])

    # (6) max charge/discharge, no simultaneous charge and discharge
    m.c6_in = pyo.Constraint(m.T, rule=lambda m, t: m.I[t] <= CD * m.C[t])
    m.c6_out = pyo.Constraint(m.T, rule=lambda m, t: m.O[t] <= CD * (1 - m.C[t]))

    # (7) range of F
    m.c7 = pyo.Constraint(m.T, rule=lambda m, t: m.F[t] <= 1)

    # (8) max purchase + sale
    m.c8 = pyo.Constraint(m.T, rule=lambda m, t: m.B[t] + m.S[t] <= G)

    # (9) terminal condition: end-of-window SOC >= initial SOC of the FIRST window
    m.c9 = pyo.Constraint(expr=m.SOC[n] >= SOC_INIT)

    return m


def get_solver():
    """'scip' = SCIP executable (conda-forge); 'scip_direct' = via pyscipopt."""
    for name in ["scip", "scip_direct"]:
        s = pyo.SolverFactory(name)
        if s.available(exception_flag=False):
            for k, v in SOLVER_OPTIONS.items():
                s.options[k] = v
            return s
    raise RuntimeError("SCIP not found (install with: conda install -c conda-forge scip)")

## 4. Rolling horizon
Each 7-day window is solved, only its first day is implemented, and the SOC at the end of that day is the initial SOC of the next window.

In [5]:
solver = get_solver()
soc0 = SOC_INIT
implemented, summary = [], []

for r in range(N_ROLLS):
    a = start_idx + r * PPD
    win = data.iloc[a: a + WINDOW_DAYS * PPD].reset_index(drop=True)
    day = win["datetime"].iloc[0].date()

    m = build_model(win, soc0)
    t0 = time.time()
    res = solver.solve(m)
    solve_time = time.time() - t0
    if res.solver.status != pyo.SolverStatus.ok:
        raise RuntimeError(f"roll {r + 1}: solver status {res.solver.status}")
    status = str(res.solver.message or res.solver.termination_condition)

    # keep only the first day of the window
    T1 = range(1, PPD + 1)
    v = lambda var: [pyo.value(var[t]) for t in T1]
    day_df = win.iloc[:PPD][["datetime", "PV", "D", "spot", "PB", "PS"]].copy()
    day_df.insert(1, "roll", r + 1)
    for name in ["B", "S", "I", "O", "F", "C", "SOC", "QU", "QL"]:
        day_df[name] = v(getattr(m, name))
    day_df["C"] = day_df["C"].round().astype(int)
    implemented.append(day_df)

    # first-day indicators
    I, O, QU, QL = day_df["I"], day_df["O"], day_df["QU"], day_df["QL"]
    deg = ((I + O).sum() / PC
           + (FC * (I ** 2 + O ** 2)).sum() / (CD * PC)
           + (CP1 * (QU + QL) + CP2 * (QU ** 2 + QL ** 2)).sum() * MAXB / PC)
    real_profit = (day_df["PS"] * day_df["S"] - day_df["PB"] * day_df["B"]).sum()
    summary.append({
        "roll": r + 1,
        "date": day,
        "real_profit [EUR]": real_profit,
        "deg [cycles]": deg,
        "deg_cost [EUR]": deg * PM / MAXC,
        "net_profit [EUR]": real_profit - deg * PM / MAXC,
        "bought [kWh]": day_df["B"].sum(),
        "sold [kWh]": day_df["S"].sum(),
        "charged [kWh]": I.sum(),
        "discharged [kWh]": O.sum(),
        "PV [kWh]": day_df["PV"].sum(),
        "PV_off [kWh]": ((1 - day_df["F"]) * day_df["PV"]).sum(),
        "demand [kWh]": day_df["D"].sum(),
        "SOC_end": day_df["SOC"].iloc[-1],
        "periods_above_MaxP": int((QU > 1e-6).sum()),
        "periods_below_MinP": int((QL > 1e-6).sum()),
        "solve_time [s]": round(solve_time, 2),
        "status": status,
    })
    print(f"roll {r + 1}/{N_ROLLS}  {day}  {status}  "
          f"net profit {summary[-1]['net_profit [EUR]']:.2f} EUR  ({solve_time:.1f} s)")

    soc0 = day_df["SOC"].iloc[-1]          # carried to the next window

roll 1/2  2025-10-01  SCIP 9.2.2\x3a optimal solution; objective 13.2186558606365; 19409 simplex iterations; 23 branching nodes  net profit 1.90 EUR  (26.3 s)
roll 2/2  2025-10-02  SCIP 9.2.2\x3a optimal solution; objective 25.5963060516031; 20370 simplex iterations; 36 branching nodes  net profit 10.41 EUR  (35.1 s)


## 5. Results
Writes `implemented` and `summary` to `Results/` and shows the daily summary.

In [6]:
impl = pd.concat(implemented, ignore_index=True)
summ = pd.DataFrame(summary)
total = summ.drop(columns=["roll", "date", "SOC_end", "status"]).sum(numeric_only=True)
total["roll"], total["date"] = "TOTAL", ""
total["SOC_end"] = summ["SOC_end"].iloc[-1]
summ = pd.concat([summ, total.to_frame().T], ignore_index=True)

RESULTS_DIR.mkdir(exist_ok=True)
out = RESULTS_DIR / f"results_{START_DATE}_{N_ROLLS}d.xlsx"
with pd.ExcelWriter(out, engine="openpyxl") as w:
    impl.to_excel(w, sheet_name="implemented", index=False)
    summ.to_excel(w, sheet_name="summary", index=False)
print(f"\nwritten {out.relative_to(HERE)}")

summ.round(2)


written Results\results_2025-10-01_2d.xlsx


,roll,date,real_profit [EUR],deg [cycles],deg_cost [EUR],net_profit [EUR],bought [kWh],sold [kWh],charged [kWh],discharged [kWh],PV [kWh],PV_off [kWh],demand [kWh],SOC_end,periods_above_MaxP,periods_below_MinP,solve_time [s],status
0,1,2025-10-01,5.661506,0.941521,3.766084,1.895422,29.173012,83.139063,45.033113,56.472692,143.373717,0.0,100.847246,0.221854,4,1,26.32,SCIP 9.2.2\x3a optimal solution; objective 13....
1,2,2025-10-02,13.460311,0.762251,3.049004,10.411307,27.253526,185.082851,45.674139,34.926842,242.20563,0.0,73.629008,0.331859,0,4,35.12,SCIP 9.2.2\x3a optimal solution; objective 25....
2,TOTAL,,19.121818,1.703772,6.815088,12.30673,56.426538,268.221913,90.707251,91.399534,385.579347,0.0,174.476254,0.331859,4.0,5.0,61.44,NaN


## Output definitions
```
OUTPUT  Results/results_<START_DATE>_<N_ROLLS>d.xlsx
Sheet 'implemented' : one row per implemented 15-min period
  roll        roll that produced this day
  PV, D       available PV, demand                      [kWh]
  spot, PB, PS  spot, purchase and selling price        [EUR/kWh]
  B, S        energy bought from / sold to the grid     [kWh]
  I, O        battery charge / discharge                [kWh]
  F           fraction of PV panels in operation (meaningless when PV = 0)
  C           1 = charging mode, 0 = discharging mode
  SOC         state of charge at the end of the period  [0,1]
  QU, QL      SOC above MaxP / below MinP               [SOC fraction]
Sheet 'summary' : one row per implemented day (= per roll), last row TOTAL
  real_profit      sum(PS*S - PB*B): net cash flow with the grid [EUR]
                   (negative = the day's electricity bill exceeded sales)
  deg              battery life consumed that day (DEG terms of that day) [cycles]
  deg_cost         deg * PM / MaxC: cost of that battery wear [EUR]
  net_profit       real_profit - deg_cost [EUR]  (sum over days = model result)
  bought, sold     sum of B, S [kWh]
  charged, discharged  sum of I, O [kWh]
  PV               available PV [kWh];  PV_off = sum((1 - F) * PV) [kWh]
  demand           sum of D [kWh]
  SOC_end          SOC at the end of the day (start of the next roll)
  periods_above_MaxP / periods_below_MinP  periods outside the preservation range
  solve_time       solver time for the 7-day window [s]
  status           SCIP message; 'gap limit reached' = within 0.01 % of optimum
  TOTAL row        sums over all days; SOC_end = last day
```